# Lesson 07 Lab — Inference Precision Layers: Weights, Activations, and KV Cache

**Puzzle:** When a model is called INT4, which tensors are actually four-bit?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Inference precision belongs to separate ledgers: persistent weights, per-step activations/workspaces, accumulators, and persistent-per-request KV cache. Weight-only INT4 normally leaves activation and accumulation formats wider.

### Core mechanism

For a standard cache, `bytes = 2 × layers × batch × sequence × kv_heads × head_dim × bytes_per_element`; the leading two is for keys and values. Grouped-query attention changes `kv_heads`, not the number of query heads.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "07-inference-precision-layers"
device = require_cuda()
torch.manual_seed(2026 + 7)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Compressing weights creates room for cache or concurrency but does not shrink every runtime object. Cache quantization may increase capacity while adding Q/DQ work and attention error.

### What this code tests

The lab validates the KV element-count formula with a live allocation and projects several context lengths without pretending to allocate a full model.

**Experiment:** Build a memory ledger and allocate representative BF16 and INT8 KV tensors on CUDA to validate element-count arithmetic.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
cfg = {"layers": 32, "kv_heads": 8, "head_dim": 128, "batch": 1}
rows = []
for seq in (2048, 8192, 32768):
    elements = 2 * cfg["layers"] * cfg["batch"] * seq * cfg["kv_heads"] * cfg["head_dim"]
    rows.append({"sequence": seq, "bf16_gib": round(elements*2/2**30, 4), "int8_gib": round(elements/2**30, 4)})
k = torch.empty(2, 4096, 8, 128, device=device, dtype=torch.bfloat16)
actual = k.numel() * k.element_size()
result = base_result(7, "pytorch-gpu"); result.update({"configuration": cfg, "projected_kv": rows,
    "allocation_probe": {"shape": list(k.shape), "bytes": actual, "dtype": str(k.dtype)},
    "conclusion": "Weights, activations, and KV cache require separate precision and memory ledger entries."})


## 3. Inspect the evidence

Report each object separately. A checkpoint-size reduction does not establish the same reduction in runtime peak memory.

### Acceptance and rollback gate

Measure allocated/reserved/peak memory separately and reconcile them with object-level arithmetic. A checkpoint byte count is not a runtime memory result.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "allocation_probe": {
    "bytes": 16777216,
    "dtype": "torch.bfloat16",
    "shape": [
      2,
      4096,
      8,
      128
    ]
  },
  "conclusion": "Weights, activations, and KV cache require separate precision and memory ledger entries.",
  "configuration": {
    "batch": 1,
    "head_dim": 128,
    "kv_heads": 8,
    "layers": 32
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:23+00:00",
  "lesson": 7,
  "projected_kv": [
    {
      "bf16_gib": 0.25,
      "int8_gib": 0.125,
      "sequence": 2048
    },
    {
      "bf16_gib": 1.0,
      "int8_gib": 0.5,
      "sequence": 8192
    },
    {
      "bf16_gib": 4.0,
      "int8_gib": 2.0,
      "sequence": 32768
    }
  ],
  "schema_version": 1
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Name the object and lifecycle whenever you name a precision: weights, activations, accumulators, or cache.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).